# Phase 2 — Normalisation validation on training data

This notebook validates `src/normalization.py` on the training data. It covers:

* before/after examples
* parse coverage
* runtime benchmarks
* diagnostics on sampled ground-truth pairs, compared with a random-pair control

**Rules**
* Training files only; the test set is never read.
* DuckDB is used for all sampling, with an in-memory database and no working file on disk.
* Samples are deterministic, selected by hash rather than by reservoir sampling.
* Python only ever processes samples of at most ~250k rows.
* The ground truth is used **only for evaluation**. No mapping is learned from it.

## 1. Setup

In [1]:
import sys, os, time, json, re, platform
from pathlib import Path
import duckdb, numpy as np, pandas as pd, psutil
from rapidfuzz import fuzz
from rapidfuzz.process import cpdist
from IPython.display import display

PROJECT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(PROJECT))
from src import normalization as N
from src.normalization import (normalize_name, normalize_address, extract_name_features,
                               extract_address_features, detect_script, transliterate_text)

pd.set_option("display.width", 250); pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 70); pd.set_option("display.max_rows", 200)

RAW = PROJECT / "data" / "raw"
EXP = PROJECT / "experiments"
F = {k: RAW / f for k, f in [("s1", "train_source1.tsv"), ("s2", "train_source2.tsv"),
                              ("s3", "train_source3.tsv"), ("gt", "train_ground_truth.tsv")]}
con = duckdb.connect()                                   # in-memory
con.execute("SET memory_limit='3GB'")
con.execute(f"SET temp_directory='{PROJECT / 'data' / 'processed' / 'duckdb_tmp'}'")
con.execute("SET max_temp_directory_size='1GB'")
src = lambda k: f"read_csv_auto('{F[k]}', delim='\t', header=true, all_varchar=true)"
q = lambda sql: con.execute(sql).df()
PROC = psutil.Process()
RESULTS, T0 = {}, time.time()
print(f"python {sys.version.split()[0]} | duckdb {duckdb.__version__} | cpus {os.cpu_count()} | RSS {PROC.memory_info().rss/1e9:.2f} GB")

python 3.13.9 | duckdb 1.5.5 | cpus 8 | RSS 0.13 GB


## 2. Deterministic record sample
`hash(entity_id) % 50 = 0` keeps about 2% of each source, roughly 250k records in total. The same rows are selected on every run.

In [2]:
t = time.time()
con.execute(f"""CREATE TABLE smp AS
  SELECT 'S1' src, * FROM {src('s1')} WHERE hash(entity_id) % 50 = 0
  UNION ALL SELECT 'S2', * FROM {src('s2')} WHERE hash(entity_id) % 50 = 0
  UNION ALL SELECT 'S3', * FROM {src('s3')} WHERE hash(entity_id) % 50 = 0""")
smp = q("SELECT * FROM smp")
print(f"{len(smp):,} sampled records in {time.time()-t:.1f}s")
display(smp.groupby(["src", "country"]).size().unstack())

250,617 sampled records in 1.5s


country,India,US
src,,
S1,17759,26456
S2,40380,60779
S3,42022,63221


## 3. Before/after examples
`core_latin` is the main cross-script key. `core` keeps the original script. `raw` is never modified.

In [3]:
NAME_COLS = ["raw", "basic", "punct", "core", "core_latin", "legal_forms", "alias_marker", "aliases", "website_label", "script"]
ADDR_COLS = ["raw", "normalized", "house_number", "house_number_core", "street", "city", "places", "state", "unit", "po_box", "noise_removed"]

def name_table(df, n=6, seed=1):
    d = df.sample(min(n, len(df)), random_state=seed)
    rows = [{"src": s, **{c: getattr(normalize_name(v), c) for c in NAME_COLS}} for s, v in zip(d.src, d.business_name)]
    return pd.DataFrame(rows)

def addr_table(df, n=6, seed=1):
    d = df.sample(min(n, len(df)), random_state=seed)
    rows = []
    for s, v, c in zip(d.src, d.business_address, d.country):
        a = normalize_address(v, c)
        rows.append({"src": s, "country": c, **{k: getattr(a, k) for k in ADDR_COLS}})
    return pd.DataFrame(rows)

INDIC = r"[\u0900-\u0DFF]"
is_indic = smp.business_name.str.contains(INDIC)
name_cases = {
    "US names": smp[(smp.country == "US")],
    "Indian names (Latin script)": smp[(smp.country == "India") & ~is_indic],
    "Messy names (junk prefix / double space / accents / OCR digit)":
        smp[smp.business_name.str.contains(r"^[^A-Za-z0-9\u0900-\u0DFF]|\s{2}|[\u00C0-\u024F]|\b[A-Za-z]*[015][A-Za-z]{3,}\b")],
    "Website names": smp[smp.business_name.str.contains(r"(?i)\.(?:com|in|net|org|co)\s*$")],
    "Legal-form variations": smp[smp.business_name.str.contains(r"(?i)\b(?:pvt|ltd|l\.l\.c|llc|inc|corp|llp|pllc|p\.c|limited)\b")],
    "Alias expressions": smp[smp.business_name.str.contains(r"(?i)\s(?:a/k/a|aka|dba|d/b/a|formerly|t/a)\b")],
}
for title, df in name_cases.items():
    print(f"\n### {title}  ({len(df):,} in sample)")
    display(name_table(df, 8 if "Legal" in title else 6))


### US names  (150,456 in sample)


,src,raw,basic,punct,core,core_latin,legal_forms,alias_marker,aliases,website_label,script
0,S3,orthopedic tri-state physicians corp,orthopedic tri-state physicians corp,orthopedic tri state physicians corp,orthopedic tri state physicians,orthopedic tri state physicians,"(corp,)",None,(),None,latin
1,S3,Banks Lane (L.L.C.),banks lane (llc),banks lane llc,banks lane,banks lane,"(llc,)",None,(),None,latin
2,S2,NR-Rio,nr-rio,nr rio,nr rio,nr rio,(),None,(),None,latin
3,S3,Secure Deep LLC,secure deep llc,secure deep llc,secure deep,secure deep,"(llc,)",None,(),None,latin
4,S2,*** board of housing llc,*** board of housing llc,board of housing llc,board of housing,board of housing,"(llc,)",None,(),None,latin
5,S3,"Jar1yx 5mart, LLC","jar1yx 5mart, llc",jar1yx 5mart llc,jarlyx smart,jarlyx smart,"(llc,)",None,(),None,latin



### Indian names (Latin script)  (85,204 in sample)


,src,raw,basic,punct,core,core_latin,legal_forms,alias_marker,aliases,website_label,script
0,S1,Star Seven Projects Private Limited,star seven projects private limited,star seven projects private limited,star seven projects,star seven projects,"(private limited,)",None,(),NaN,latin
1,S2,Smt Sai Consultancy,smt sai consultancy,smt sai consultancy,smt sai consultancy,smt sai consultancy,(),None,(),NaN,latin
2,S2,Marketing Sinewave Services,marketing sinewave services,marketing sinewave services,marketing sinewave services,marketing sinewave services,(),None,(),NaN,latin
3,S2,ílluminatifinvest.com,ílluminatifinvest.com,illuminatifinvest com,illuminatifinvest,illuminatifinvest,(),None,(),illuminatifinvest,latin
4,S1,Smart Consulting,smart consulting,smart consulting,smart consulting,smart consulting,(),None,(),NaN,latin
5,S3,Continental Properties Private Limited,continental properties private limited,continental properties private limited,continental properties,continental properties,"(private limited,)",None,(),NaN,latin



### Messy names (junk prefix / double space / accents / OCR digit)  (41,063 in sample)


...

,src,raw,basic,punct,core,core_latin,legal_forms,alias_marker,aliases,website_label,script
0,S2,AAAINDIA.COM,aaaindia.com,aaaindia com,aaaindia,aaaindia,(),None,(),aaaindia,latin
1,S2,kincaidursulinap.com,kincaidursulinap.com,kincaidursulinap com,kincaidursulinap,kincaidursulinap,(),None,(),kincaidursulinap,latin
2,S2,vamsifresh.com,vamsifresh.com,vamsifresh com,vamsifresh,vamsifresh,(),None,(),vamsifresh,latin
3,S2,clearbarings.com,clearbarings.com,clearbarings com,clearbarings,clearbarings,(),None,(),clearbarings,latin
4,S2,dermatology.com,dermatology.com,dermatology com,dermatology,dermatology,(),None,(),dermatology,latin
5,S2,soham.com,soham.com,soham com,soham,soham,(),None,(),soham,latin



### Legal-form variations  (121,781 in sample)


,src,raw,basic,punct,core,core_latin,legal_forms,alias_marker,aliases,website_label,script
0,S1,"Select Pioneer Drilling, LLC","select pioneer drilling, llc",select pioneer drilling llc,select pioneer drilling,select pioneer drilling,"(llc,)",None,(),None,latin
1,S1,Rajpura Assurance Public Limited,rajpura assurance public limited,rajpura assurance public limited,rajpura assurance,rajpura assurance,"(public limited,)",None,(),None,latin
2,S2,Enhanced Preferred Inc,enhanced preferred inc,enhanced preferred inc,enhanced preferred,enhanced preferred,"(inc,)",None,(),None,latin
3,S2,"Adey Wright, Esq. Lakeside LLC","adey wright, esq lakeside llc",adey wright esq lakeside llc,adey wright esq lakeside,adey wright esq lakeside,"(llc,)",None,(),None,latin
4,S1,Jaymee Lemos Supreme Games PLLC,jaymee lemos supreme games pllc,jaymee lemos supreme games pllc,jaymee lemos supreme games,jaymee lemos supreme games,"(pllc,)",None,(),None,latin
5,S3,Pvt Uttam Mánagement Ltd,pvt uttam mánagement ltd,pvt uttam management ltd,uttam management,uttam management,"(private limited,)",None,(),None,latin
6,S3,Regional College P.L.L.C.,regional college pllc,regional college pllc,regional college,regional college,"(pllc,)",None,(),None,latin
7,S3,MK Cea Midtown LLC,mk cea midtown llc,mk cea midtown llc,mk cea midtown,mk cea midtown,"(llc,)",None,(),None,latin



### Alias expressions  (2,071 in sample)


,src,raw,basic,punct,core,core_latin,legal_forms,alias_marker,aliases,website_label,script
0,S3,Evodovasol aka Meentech Services Private Limited,evodovasol aka meentech services private limited,evodovasol aka meentech services private limited,evodovasol meentech services,evodovasol meentech services,"(private limited,)",aka,"(evodovasol, meentech services)",None,latin
1,S3,Onyxvantagegild+ Formerly Styles All Realty Care,onyxvantagegild+ formerly styles all realty care,onyxvantagegild formerly styles all realty care,onyxvantagegild styles all realty care,onyxvantagegild styles all realty care,(),formerly,"(onyxvantagegild, styles all realty care)",None,latin
2,S3,Tavojaxyuma aka Dream Investment Private Limited,tavojaxyuma aka dream investment private limited,tavojaxyuma aka dream investment private limited,tavojaxyuma dream investment,tavojaxyuma dream investment,"(private limited,)",aka,"(tavojaxyuma, dream investment)",None,latin
3,S3,"Fluxriza DBA: Supreme Kpet, Corp","fluxriza dba: supreme kpet, corp",fluxriza dba supreme kpet corp,fluxriza supreme kpet,fluxriza supreme kpet,"(corp,)",dba,"(fluxriza, supreme kpet)",None,latin
4,S3,"Onyxveo d/b/a Barreras, Madeline, D.O.","onyxveo d/b/a barreras, madeline, do",onyxveo d b a barreras madeline do,onyxveo barreras madeline do,onyxveo barreras madeline do,(),d/b/a,"(onyxveo, barreras madeline do)",None,latin
5,S3,Fluxiri formerly Shaneshwar Technologies Private Limited,fluxiri formerly shaneshwar technologies private limited,fluxiri formerly shaneshwar technologies private limited,fluxiri shaneshwar technologies,fluxiri shaneshwar technologies,"(private limited,)",formerly,"(fluxiri, shaneshwar technologies)",None,latin


In [4]:
print("### Indic-script names — one per script")
rows = []
for script, (lo, hi) in {"devanagari": ("\u0900", "\u097F"), "bengali": ("\u0980", "\u09FF"), "gurmukhi": ("\u0A00", "\u0A7F"),
                         "gujarati": ("\u0A80", "\u0AFF"), "oriya": ("\u0B00", "\u0B7F"), "tamil": ("\u0B80", "\u0BFF"),
                         "telugu": ("\u0C00", "\u0C7F"), "kannada": ("\u0C80", "\u0CFF"), "malayalam": ("\u0D00", "\u0D7F")}.items():
    d = smp[smp.business_name.str.contains(f"[{lo}-{hi}]")]
    rows.append(name_table(d, 2, seed=3).assign(expected_script=script))
display(pd.concat(rows, ignore_index=True)[["expected_script", "src", "raw", "core", "core_latin", "legal_forms", "script"]])

### Indic-script names — one per script


,expected_script,src,raw,core,core_latin,legal_forms,script
0,devanagari,S2,अल विजय एनर्जी प्रा. लि.,अल विजय एनर्जी,al vijay enarji,"(private limited,)",indic
1,devanagari,S2,गोल्डन एक्सपोर्ट्स प्रा. लि.,गोल्डन एक्सपोर्ट्स,goldan eksaports,"(private limited,)",indic
2,bengali,S3,গুরু টেকনোলজি প্রাইভেট লিমিটেড,গুরু টেকনোলজি,guru teknolji,"(private limited,)",indic
3,bengali,S2,শিব ফুড Private Limited,শিব ফুড,shib phud,"(private limited,)",mixed
4,gurmukhi,S3,ਗਲੋਬਲ ਪ੍ਰੋਡਕਟਸ ਪ੍ਰਾ. ਲਿ.,ਗਲੋਬਲ ਪ੍ਰੋਡਕਟਸ,galobal prodaktas,"(private limited,)",indic
5,gurmukhi,S2,ਅਰਿਹੰਤ ਇੰਡਸਟ੍ਰੀਜ਼ ਪ੍ਰਾਈਵੇਟ ਲਿਮਟਿਡ,ਅਰਿਹੰਤ ਇੰਡਸਟ੍ਰੀਜ਼,arihant indasatriz,"(private limited,)",indic
6,gujarati,S3,શિવા ઇન્ફ્રા પ્રાઇવેટ લિમિટેડ,શિવા ઇન્ફ્રા,shiva inphra,"(private limited,)",indic
7,gujarati,S3,સાઉથ ઇન્ડસ્ટ્રીઝ,સાઉથ ઇન્ડસ્ટ્રીઝ,sauth indastrijh,(),indic
8,oriya,S2,ଟେକ୍ ଇନଭେଷ୍ଟମେଣ୍ଟ୍ ପ୍ରାଇଭେଟ୍ ଲିମିଟେଡ୍,ଟେକ୍ ଇନଭେଷ୍ଟମେଣ୍ଟ୍,tek inbheshtament,"(private limited,)",indic
9,oriya,S2,ରୟାଲ୍ ଅଲ୍ ଏନର୍ଜି ଏଲ୍‌ଏଲ୍‌ପି,ରୟାଲ୍ ଅଲ୍ ଏନର୍ଜି,rayal al enarji,"(llp,)",indic


In [5]:
addr_cases = {
    "US addresses": smp[(smp.country == "US") & smp.business_address.notna()],
    "Indian addresses": smp[(smp.country == "India") & smp.business_address.notna()],
    "Messy addresses (## / null / N/A / zero-padded / native-script state)":
        smp[smp.business_address.fillna("").str.contains(r"##|(?i:\bnull\b)|N/A|<NULL>|\b0\d{2,}\b|[\u0900-\u0DFF]")],
    "Missing addresses": smp[smp.business_address.isna()],
}
for title, df in addr_cases.items():
    print(f"\n### {title}  ({len(df):,} in sample)")
    display(addr_table(df, 8 if "Messy" in title else 6))


### US addresses  (146,061 in sample)


,src,country,raw,normalized,house_number,house_number_core,street,city,places,state,unit,po_box,noise_removed
0,S3,US,"737A Freedom Ln, Grayson County, Virginia","737a freedom ln, grayson county, va",737a,737,freedom ln,grayson county,"(grayson county,)",va,NaN,None,0
1,S3,US,"80 Main St, # Unit 5, Trappe Borough, Pennsylvania","80 main st, unit 5, trappe borough, pa",80,80,main st,trappe borough,"(trappe borough,)",pa,unit 5,None,0
2,S3,US,"35 Closter Road, Orangetown, New York","35 closter rd, orangetown, ny",35,35,closter rd,orangetown,"(orangetown,)",ny,NaN,None,0
3,S3,US,"13722 Meisterwood Drive, Houston, Texas","13722 meisterwood dr, houston, tx",13722,13722,meisterwood dr,houston,"(houston,)",tx,NaN,None,0
4,S1,US,"1434 Plum Street, Springfield, OH","1434 plum st, springfield, oh",1434,1434,plum st,springfield,"(springfield,)",oh,NaN,None,0
5,S3,US,"108 Glass Avenue, Hopkinsville, Kentucky","108 glass ave, hopkinsville, ky",108,108,glass ave,hopkinsville,"(hopkinsville,)",ky,NaN,None,0



### Indian addresses  (97,757 in sample)


,src,country,raw,normalized,house_number,house_number_core,street,city,places,state,unit,po_box,noise_removed
0,S3,India,"No.621, 1St Floor, 5Th 'A'Mainroad, Ii Block Rt. Nagar, Bangalore-...","no 621, 1st fl, 5th amainroad, ii block rt nagar, bangalore-32, ba...",621,621,NaN,bangalore,"(ii block rt nagar, bangalore)",ka,NaN,None,0
1,S3,India,"Door No 642 3/470-14, Vandanam, Don Bosco Chaitanya Lane, Monvila,...","door no 642 3/470-14, vandanam, don bosco chaitanya ln, monvila, k...",642,642,don bosco chaitanya ln,trivandrum,"(vandanam, monvila, kulathoor p o, thiruvananthapuram, trivandrum)",kl,NaN,None,0
2,S1,India,"Fl No. 203, Sr No. 296/2, Building Lotus, Park Springs, Sant Tukar...","fl no 203, sr no 296/2, bldg lotus, park springs, sant tukaram nag...",203,203,sant tukaram nagar rd,pune,"(park springs, pune)",mh,bldg lotus,None,0
3,S2,India,"NO 3/250 BHARATHIYAR NAGAR AVALAPALLI ROAD, HOSUR, KRISHNAGIRI, Ta...","no 3/250 bharathiyar nagar avalapalli rd, hosur, krishnagiri, tn",3/250,3,bharathiyar nagar avalapalli rd,krishnagiri,"(hosur, krishnagiri)",tn,NaN,None,0
4,S2,India,"2-A16, ISHA MISTY GREEN CHANNASANDRA WHITE FIELD, BANGALORE, Karna...","2-a16, isha misty green channasandra white field, bangalore, ka",2-a16,2,NaN,bangalore,"(isha misty green channasandra white field, bangalore)",ka,NaN,None,0
5,S3,India,"###114-115, Kamrej, Surat, ગુજરાત","114-115, kamrej, surat, gj",114-115,114,NaN,surat,"(kamrej, surat)",gj,NaN,None,1



### Messy addresses (## / null / N/A / zero-padded / native-script state)  (38,103 in sample)


,src,country,raw,normalized,house_number,house_number_core,street,city,places,state,unit,po_box,noise_removed
0,S3,US,"02525 Morrisville Pkwy, Cary, North Carolina","2525 morrisville pkwy, cary, nc",2525,2525,morrisville pkwy,cary,"(cary,)",nc,None,None,0
1,S2,US,"002036 MARY ELLA DRIVE, NULL, GEORGETOWN, IN","2036 mary ella dr, georgetown, in",2036,2036,mary ella dr,georgetown,"(georgetown,)",in,None,None,1
2,S2,US,"003177 DOME ROCK PL, PRESCOTT, AZ","3177 dome rock pl, prescott, az",3177,3177,dome rock pl,prescott,"(prescott,)",az,None,None,0
3,S3,India,"C 35 First Floor, Nwa Punjabi Bagh Clubroad, West Delhi, New Delhi...","c 35 1st fl, nwa punjabi bagh clubroad, w delhi, new delhi, dl",NaN,NaN,NaN,new delhi,"(nwa punjabi bagh clubroad, w delhi, new delhi)",dl,None,None,0
4,S3,India,"ಕರ್ನಾಟಕ, Bangalore South, Bangalore, 74, 15Th Cross, J P Nagar, 3R...","ka, bangalore s, bangalore, 74, 15th cross, j p nagar, 3rd phase b...",74,74,NaN,j p nagar,"(bangalore s, bangalore, j p nagar)",ka,None,None,0
5,S2,India,"NO - 13 PADMAVAHI COLONY, BALAJI HILLS, UPPAL, తెలంగాణ","no 13 padmavahi colony, balaji hills, uppal, tg",13,13,padmavahi colony,uppal,"(balaji hills, uppal)",tg,None,None,0
6,S3,India,"Floor Shop-301 Silk Heritage Textile Market, Near Gautam Market, R...","fl shop-301 silk heritage textile market, nr gautam market, ring r...",NaN,NaN,ring rd,sachin,"(nr gautam market, surat, sachin)",gj,None,None,0
7,S2,India,"HOUSE NO. 662, SECTOR-10, PANCHKULA, हरियाणा","h no 662, sector-10, panchkula, hr",662,662,NaN,panchkula,"(panchkula,)",hr,None,None,0



### Missing addresses  (6,799 in sample)


,src,country,raw,normalized,house_number,house_number_core,street,city,places,state,unit,po_box,noise_removed
0,S2,India,None,,None,None,None,None,(),None,None,None,0
1,S2,US,None,,None,None,None,None,(),None,None,None,0
2,S3,US,None,,None,None,None,None,(),None,None,None,0
3,S3,India,None,,None,None,None,None,(),None,None,None,0
4,S2,US,None,,None,None,None,None,(),None,None,None,0
5,S3,US,None,,None,None,None,None,(),None,None,None,0


## 4. Parse coverage on the full 250k sample

In [6]:
t = time.time()
nf = pd.DataFrame([extract_name_features(v) for v in smp.business_name])
af = pd.DataFrame([extract_address_features(a, c) for a, c in zip(smp.business_address, smp.country)])
cov_src = pd.concat([smp[["src", "country"]].reset_index(drop=True), nf, af], axis=1)
print(f"features for {len(cov_src):,} records in {time.time()-t:.1f}s")
agg = cov_src.groupby(["src", "country"]).agg(
    n=("name_script", "size"),
    pct_name_indic=("name_is_indic", "mean"), pct_name_mixed=("name_script", lambda s: (s == "mixed").mean()),
    pct_legal_form=("name_has_legal_form", "mean"), pct_website=("name_is_website", "mean"),
    pct_alias=("name_has_alias", "mean"), pct_ocr_digit=("name_has_ocr_digit", "mean"),
    pct_addr_missing=("addr_is_missing", "mean"), pct_state=("addr_has_state", "mean"),
    pct_house_number=("addr_has_house_number", "mean"), pct_city=("addr_city", lambda s: (s != "").mean()),
    pct_street=("addr_street", lambda s: (s != "").mean()), pct_noise_removed=("addr_noise_removed", lambda s: (s > 0).mean()),
)
agg.iloc[:, 1:] = (100 * agg.iloc[:, 1:]).round(2)
display(agg)
print("Legal forms detected (top 12):")
display(cov_src.name_legal_forms.replace("", np.nan).dropna().value_counts().head(12).to_frame().T)
print("Alias markers:"); display(cov_src.name_alias_marker.replace("", np.nan).dropna().value_counts().to_frame().T)
print("States resolved (per country, top 10):")
display(cov_src[cov_src.addr_state != ""].groupby("country").addr_state.value_counts().groupby(level=0).head(10).unstack(0))
# records that have an address but no state resolved - what is the last component?
nost = smp[(cov_src.addr_state == "").values & smp.business_address.notna().values]
print(f"addresses with no state resolved: {len(nost):,}; most common last components:")
display(nost.business_address.str.split(",").str[-1].str.strip().value_counts().head(12).to_frame().T)
RESULTS["coverage"] = agg.reset_index().to_dict(orient="records")

features for 250,617 records in 28.7s


n  pct_name_indic  pct_name_mixed  pct_legal_form  pct_website  pct_alias  pct_ocr_digit  pct_addr_missing  pct_state  pct_house_number  pct_city  pct_street  pct_noise_removed
src country                                                                                                                                                                                      
S1  India    17759            0.00            0.00           84.24         0.00       0.00           0.00              0.00     100.00             79.52     99.90       58.40               0.04
    US       26456            0.00            0.00           57.52         0.00       0.00           0.00              0.00     100.00             99.86     99.44       96.91               0.00
S2  India    40380           23.31            0.91           75.39         2.94       0.00           1.67              2.81      97.19             78.51     96.05       56.03               5.89
    US       60779            0.00            0.00           49.31         3.98       0.00           2.50              3.59      96.41             90.13     95.89       93.23               6.36
S3  India    42022           13.20            1.67           72.56         2.91       2.29           1.93              3.02      96.98             75.95     96.62       50.07               5.26
    US       63221            0.00            0.00           48.59         3.76       3.51           2.54              3.50      96.50             90.65     95.92       93.14               5.93

Legal forms detected (top 12):


name_legal_forms,private limited,llc,inc,limited,corp,co,private,llp,pc,lp,pllc,public limited
count,46433,32155,22509,16491,8884,6075,5955,4938,2630,1921,1575,1081


Alias markers:


name_alias_marker,dba,formerly,f/k/a,doing business as,d/b/a,t/a,aka,a/k/a,trading as,formerly known as,fka
count,814,478,296,255,225,218,199,197,196,152,151


States resolved (per country, top 10):


country,India,US
addr_state,,
mh,21089.0,NaN
dl,13217.0,NaN
up,8098.0,NaN
ka,7928.0,NaN
tn,6945.0,6492.0
wb,6443.0,NaN
gj,6428.0,NaN
tg,5233.0,NaN
kl,3932.0,NaN


addresses with no state resolved: 0; most common last components:


business_address
count


## 5. Runtime benchmarks
Every benchmark runs on the full 250k-record sample, single process, starting from a cold cache. The last step checks how well the work scales across processes with `multiprocessing` (8 workers).

In [7]:
def bench(label, fn, items):
    t = time.perf_counter(); out = [fn(*it) if isinstance(it, tuple) else fn(it) for it in items]; dt = time.perf_counter() - t
    return {"step": label, "n": len(items), "seconds": round(dt, 2), "us_per_record": round(1e6 * dt / len(items), 1)}, out

names = smp.business_name.tolist()
addrs = list(zip(smp.business_address.tolist(), smp.country.tolist()))
indic_names = [n for n in names if re.search(INDIC, n or "")]
rss0 = PROC.memory_info().rss
N._normalize_name_cached.cache_clear(); N._normalize_address_cached.cache_clear(); N._translit_token.cache_clear()
bm = []
r, _ = bench("normalize_name (cold cache)", normalize_name, names); bm.append(r)
r, _ = bench("normalize_name (warm cache, same records)", normalize_name, names); bm.append(r)
r, _ = bench("normalize_address (cold cache)", lambda a, c: normalize_address(a, c), addrs); bm.append(r)
N._translit_token.cache_clear()
r, _ = bench("transliterate_text (Indic names only, cold)", transliterate_text, indic_names); bm.append(r)
r, _ = bench("extract_name_features", extract_name_features, names); bm.append(r)
r, _ = bench("extract_address_features", lambda a, c: extract_address_features(a, c), addrs); bm.append(r)
rss1 = PROC.memory_info().rss
bm = pd.DataFrame(bm); display(bm)
print(f"cache sizes: names {N._normalize_name_cached.cache_info().currsize:,}, addresses {N._normalize_address_cached.cache_info().currsize:,}"
      f" | RSS growth {(rss1-rss0)/1e6:,.0f} MB")
RESULTS["benchmark_single_process"] = bm.to_dict(orient="records")
RESULTS["benchmark_rss_growth_mb"] = round((rss1 - rss0) / 1e6)

,step,n,seconds,us_per_record
0,normalize_name (cold cache),250617,9.02,36.0
1,"normalize_name (warm cache, same records)",250617,0.06,0.2
2,normalize_address (cold cache),250617,12.73,50.8
3,"transliterate_text (Indic names only, cold)",14957,0.16,10.9
4,extract_name_features,250617,5.09,20.3
5,extract_address_features,250617,0.74,2.9


cache sizes: names 242,361, addresses 243,153 | RSS growth -81 MB


In [8]:
# Multiprocessing scaling test (8 workers, chunks of 5k records) using the batch API.
import multiprocessing as mp
from src.normalization import normalize_records_chunk
chunks = [(names[i:i + 5000], [a for a, _ in addrs[i:i + 5000]], [c for _, c in addrs[i:i + 5000]]) for i in range(0, len(names), 5000)]
t = time.perf_counter()
with mp.get_context("spawn").Pool(8) as pool:
    counts = [len(r) for r in pool.map(normalize_records_chunk, chunks)]   # results are pickled back (realistic)
dt = time.perf_counter() - t
mp_row = {"step": "name + address, 8 processes (incl. pool start-up)", "n": sum(counts), "seconds": round(dt, 2),
          "us_per_record": round(1e6 * dt / sum(counts), 1)}
display(pd.DataFrame([mp_row]))
single = bm.set_index("step").loc[["normalize_name (cold cache)", "normalize_address (cold cache)"], "us_per_record"].sum()
total_records = 2_206_821 + 5_034_616 + 5_285_603
extrap = pd.DataFrame([
    {"setting": "single process, cold cache (measured rate)", "us_per_record": single, "est_minutes_all_12.5M": round(single * total_records / 6e7, 1)},
    {"setting": "8 processes (measured rate)", "us_per_record": mp_row["us_per_record"], "est_minutes_all_12.5M": round(mp_row["us_per_record"] * total_records / 6e7, 1)},
])
display(extrap)
RESULTS["benchmark_multiprocess"] = mp_row
RESULTS["benchmark_extrapolation"] = extrap.to_dict(orient="records")

,step,n,seconds,us_per_record
0,"name + address, 8 processes (incl. pool start-up)",250617,6.63,26.4


,setting,us_per_record,est_minutes_all_12.5M
0,"single process, cold cache (measured rate)",86.8,18.1
1,8 processes (measured rate),26.4,5.5


## 6. True-pair diagnostics
About 51k true pairs are sampled deterministically with `hash(s1_id, match_id) % 150 = 7`, and both sides are normalised. As a **control**, the same targets are re-paired at random with other S1 records from the same country. A good representation raises agreement on true pairs **without** raising it much on random pairs.

The ground truth is used only to score these comparisons.

In [9]:
t = time.time()
con.execute(f"""CREATE TABLE sp AS
  SELECT s1_id, match_id FROM (
    SELECT source1_entity_id s1_id, trim(m) match_id
    FROM {src('gt')}, unnest(string_split(matched_entity_ids, ',')) AS u(m)
    WHERE matched_entity_ids IS NOT NULL)
  WHERE hash(s1_id, match_id) % 150 = 7""")
con.execute(f"CREATE TABLE a AS SELECT * FROM {src('s1')} WHERE entity_id IN (SELECT s1_id FROM sp)")
con.execute(f"""CREATE TABLE b AS SELECT 'S2' src, * FROM {src('s2')} WHERE entity_id IN (SELECT match_id FROM sp)
                UNION ALL SELECT 'S3', * FROM {src('s3')} WHERE entity_id IN (SELECT match_id FROM sp)""")
pairs = q("""SELECT sp.s1_id, sp.match_id, b.src, a.country,
                    a.business_name a_name, a.business_address a_addr, b.business_name b_name, b.business_address b_addr
             FROM sp JOIN a ON a.entity_id = sp.s1_id JOIN b ON b.entity_id = sp.match_id""")
print(f"{len(pairs):,} true pairs sampled in {time.time()-t:.1f}s")
rng = np.random.default_rng(2026)
ctrl = pairs.copy()
for c in ctrl.country.unique():                           # random re-pairing within country
    idx = np.where(ctrl.country == c)[0]
    perm = rng.permutation(idx)
    ctrl.loc[idx, ["a_name", "a_addr"]] = pairs.loc[perm, ["a_name", "a_addr"]].values
ctrl = ctrl[ctrl.a_name.values != pairs.b_name.values]    # drop accidental identical-name pairs
display(pairs.groupby(["src", "country"]).size().unstack())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

50,934 true pairs sampled in 4.0s


country,India,US
src,,
S2,9857,14849
S3,10597,15631


In [10]:
def jacc(x, y):
    x, y = set(x), set(y)
    return len(x & y) / len(x | y) if (x or y) else np.nan

def diagnostics(df):
    A = [normalize_name(v) for v in df.a_name]; B = [normalize_name(v) for v in df.b_name]
    AA = [normalize_address(v, c) for v, c in zip(df.a_addr, df.country)]
    BA = [normalize_address(v, c) for v, c in zip(df.b_addr, df.country)]
    o = pd.DataFrame({"src": df.src.values, "country": df.country.values})
    o["script_mismatch"] = [(x.script in ("indic", "mixed")) != (y.script in ("indic", "mixed")) for x, y in zip(A, B)]
    # ---- names
    o["name_raw_eq"] = (df.a_name.values == df.b_name.values)
    o["name_basic_eq"] = [x.basic == y.basic for x, y in zip(A, B)]
    o["name_punct_eq"] = [x.punct == y.punct for x, y in zip(A, B)]
    o["name_core_eq"] = [x.core == y.core and x.core != "" for x, y in zip(A, B)]
    o["name_core_latin_eq"] = [x.core_latin == y.core_latin and x.core_latin != "" for x, y in zip(A, B)]
    # any shared key among (core_latin, alias sides, website label), or equal compact forms, or one side's
    # website label equal to the other side's compact core (wenonahsmetalworks.com ~ Wenonah's Metal Works)
    o["name_variant_eq"] = [bool(set(x.variants) & set(y.variants)) or bool(x.compact and x.compact == y.compact)
                            or bool(x.website_label and x.website_label == y.compact)
                            or bool(y.website_label and y.website_label == x.compact) for x, y in zip(A, B)]
    o["name_legal_form_eq"] = [x.legal_forms == y.legal_forms for x, y in zip(A, B)]
    o["name_shared_core_token"] = [bool(set(x.core_latin.split()) & set(y.core_latin.split())) for x, y in zip(A, B)]
    o["name_shared_raw_token"] = [bool(set(re.findall(r"\w+", (p or "").lower())) & set(re.findall(r"\w+", (q_ or "").lower())))
                                  for p, q_ in zip(df.a_name, df.b_name)]
    o["name_tset_raw"] = cpdist(df.a_name.fillna("").str.lower().tolist(), df.b_name.fillna("").str.lower().tolist(), scorer=fuzz.token_set_ratio, workers=-1)
    o["name_tset_basic"] = cpdist([x.basic for x in A], [y.basic for y in B], scorer=fuzz.token_set_ratio, workers=-1)
    o["name_tset_core_latin"] = cpdist([x.core_latin for x in A], [y.core_latin for y in B], scorer=fuzz.token_set_ratio, workers=-1)
    o["name_phonetic_ratio"] = cpdist([x.phonetic for x in A], [y.phonetic for y in B], scorer=fuzz.token_set_ratio, workers=-1)
    # ---- addresses
    both = np.array([not (x.is_missing or y.is_missing) for x, y in zip(AA, BA)])
    o["addr_both_present"] = both
    o["addr_basic_eq"] = [x.basic == y.basic for x, y in zip(AA, BA)]
    o["addr_normalized_eq"] = [x.normalized == y.normalized for x, y in zip(AA, BA)]
    o["addr_component_jaccard"] = [jacc(x.components, y.components) for x, y in zip(AA, BA)]
    o["addr_shared_component"] = [bool(set(x.components) & set(y.components)) for x, y in zip(AA, BA)]
    o["addr_token_jaccard"] = [jacc(x.tokens, y.tokens) for x, y in zip(AA, BA)]
    o["addr_tset_raw"] = cpdist(df.a_addr.fillna("").str.lower().tolist(), df.b_addr.fillna("").str.lower().tolist(), scorer=fuzz.token_set_ratio, workers=-1)
    o["addr_tset_normalized"] = cpdist([x.normalized for x in AA], [y.normalized for y in BA], scorer=fuzz.token_set_ratio, workers=-1)
    def agree(f):
        return [None if (getattr(x, f) is None or getattr(y, f) is None) else getattr(x, f) == getattr(y, f) for x, y in zip(AA, BA)]
    for f in ["house_number_core", "house_number", "state", "city", "street"]:
        o[f"addr_{f}_agree"] = agree(f)
    o["addr_shared_place"] = [None if not (x.places and y.places) else bool(set(x.places) & set(y.places)) for x, y in zip(AA, BA)]
    o["addr_shared_number"] = [None if not (x.numbers and y.numbers) else bool(set(x.numbers) & set(y.numbers)) for x, y in zip(AA, BA)]
    for c in ["addr_tset_raw", "addr_tset_normalized", "addr_component_jaccard", "addr_token_jaccard"]:
        o.loc[~both, c] = np.nan
    return o

t = time.time()
dt_true = diagnostics(pairs); dt_ctrl = diagnostics(ctrl)
print(f"diagnostics on {len(dt_true):,} true + {len(dt_ctrl):,} control pairs in {time.time()-t:.1f}s")

diagnostics on 50,934 true + 50,934 control pairs in 10.9s


In [11]:
BOOL = ["name_raw_eq", "name_basic_eq", "name_punct_eq", "name_core_eq", "name_core_latin_eq", "name_variant_eq",
        "name_legal_form_eq", "name_shared_raw_token", "name_shared_core_token",
        "addr_basic_eq", "addr_normalized_eq", "addr_shared_component",
        "addr_house_number_core_agree", "addr_house_number_agree", "addr_state_agree", "addr_city_agree", "addr_shared_place", "addr_street_agree", "addr_shared_number"]
NUM = ["name_tset_raw", "name_tset_basic", "name_tset_core_latin", "name_phonetic_ratio",
       "addr_tset_raw", "addr_tset_normalized", "addr_component_jaccard", "addr_token_jaccard"]

def summarise(o):
    r = {}
    for c in BOOL:
        s = o[c].dropna().astype(bool)
        r[c] = round(100 * s.mean(), 2) if len(s) else np.nan
    for c in NUM:
        r[c + " (mean)"] = round(o[c].mean(), 1)
    r["n"] = len(o)
    return pd.Series(r)

tab = pd.DataFrame({"true: all": summarise(dt_true), "true: S2": summarise(dt_true[dt_true.src == "S2"]),
                    "true: S3": summarise(dt_true[dt_true.src == "S3"]), "true: US": summarise(dt_true[dt_true.country == "US"]),
                    "true: India": summarise(dt_true[dt_true.country == "India"]),
                    "control: random pairs": summarise(dt_ctrl)})
tab["lift (true all - control)"] = (tab["true: all"] - tab["control: random pairs"]).round(2)
print("Boolean rows = % of pairs (agreement fields: % of pairs where both sides have the field). Numeric rows = mean score.")
display(tab)
RESULTS["true_pair_diagnostics"] = tab.reset_index().rename(columns={"index": "metric"}).to_dict(orient="records")

Boolean rows = % of pairs (agreement fields: % of pairs where both sides have the field). Numeric rows = mean score.


,true: all,true: S2,true: S3,true: US,true: India,control: random pairs,lift (true all - control)
name_raw_eq,4.710000,4.970000,4.460000,6.080000,2.660000,0.000000,4.71
name_basic_eq,17.520000,17.830000,17.230000,21.330000,11.840000,0.000000,17.52
name_punct_eq,26.680000,26.290000,27.060000,31.980000,18.790000,0.000000,26.68
name_core_eq,52.150000,53.080000,51.270000,56.700000,45.360000,0.010000,52.14
name_core_latin_eq,52.310000,53.250000,51.420000,56.700000,45.760000,0.010000,52.30
name_variant_eq,58.300000,57.120000,59.410000,63.180000,51.030000,0.010000,58.29
name_legal_form_eq,71.380000,70.710000,72.010000,72.750000,69.350000,32.300000,39.08
name_shared_raw_token,85.830000,84.050000,87.510000,91.990000,76.650000,15.650000,70.18
name_shared_core_token,87.960000,86.970000,88.900000,91.820000,82.210000,1.530000,86.43
addr_basic_eq,7.420000,10.720000,4.310000,7.990000,6.580000,0.000000,7.42


In [12]:
print("### Indic-vs-Latin name subset (script mismatch between S1 and target)")
sub_t, sub_c = dt_true[dt_true.script_mismatch], dt_ctrl[dt_ctrl.script_mismatch]
cols = ["name_shared_raw_token", "name_shared_core_token", "name_core_latin_eq", "name_legal_form_eq",
        "name_tset_raw", "name_tset_basic", "name_tset_core_latin", "name_phonetic_ratio"]
mm = pd.DataFrame({"true (script mismatch)": summarise(sub_t), "control (script mismatch)": summarise(sub_c)}).loc[
    [c if c in BOOL else c + " (mean)" for c in cols] + ["n"]]
display(mm)
q90 = pd.DataFrame({
    "tset_raw >= 80": [round(100 * (sub_t.name_tset_raw >= 80).mean(), 2), round(100 * (sub_c.name_tset_raw >= 80).mean(), 2)],
    "tset_core_latin >= 80": [round(100 * (sub_t.name_tset_core_latin >= 80).mean(), 2), round(100 * (sub_c.name_tset_core_latin >= 80).mean(), 2)],
    "phonetic >= 80": [round(100 * (sub_t.name_phonetic_ratio >= 80).mean(), 2), round(100 * (sub_c.name_phonetic_ratio >= 80).mean(), 2)],
}, index=["true", "control"])
display(q90)
RESULTS["script_mismatch_subset"] = {"table": mm.reset_index().rename(columns={"index": "metric"}).to_dict(orient="records"),
                                     "thresholds": q90.reset_index().rename(columns={"index": "set"}).to_dict(orient="records")}

### Indic-vs-Latin name subset (script mismatch between S1 and target)


,true (script mismatch),control (script mismatch)
name_shared_raw_token,7.960000,2.090000
name_shared_core_token,40.040000,0.360000
name_core_latin_eq,3.330000,0.000000
name_legal_form_eq,100.000000,50.100000
name_tset_raw (mean),14.100000,10.300000
name_tset_basic (mean),14.200000,10.300000
name_tset_core_latin (mean),70.500000,30.700001
name_phonetic_ratio (mean),94.300003,39.500000
n,3629.000000,3629.000000


,tset_raw >= 80,tset_core_latin >= 80,phonetic >= 80
true,2.31,30.37,97.80
control,0.00,0.11,0.36


In [13]:
print("### Examples: Indic vs Latin true pairs after transliteration")
ex = pairs[dt_true.script_mismatch.values].head(10)
display(pd.DataFrame([{"S1 raw": a, "target raw": b, "S1 core_latin": normalize_name(a).core_latin,
                       "target core_latin": normalize_name(b).core_latin,
                       "phonetic S1": normalize_name(a).phonetic, "phonetic target": normalize_name(b).phonetic}
                      for a, b in zip(ex.a_name, ex.b_name)]))
print("### Examples: true pairs whose core_latin still disagrees (same script)")
bad = pairs[(~dt_true.name_core_latin_eq.values) & (~dt_true.script_mismatch.values)].sample(10, random_state=4)
display(pd.DataFrame([{"S1": normalize_name(a).core_latin, "target": normalize_name(b).core_latin,
                       "tset": round(fuzz.token_set_ratio(normalize_name(a).core_latin, normalize_name(b).core_latin))}
                      for a, b in zip(bad.a_name, bad.b_name)]))
print("### Examples: address structure on true pairs")
exa = pairs[pairs.a_addr.notna() & pairs.b_addr.notna()].sample(8, random_state=5)
rows = []
for a, b, c in zip(exa.a_addr, exa.b_addr, exa.country):
    x, y = normalize_address(a, c), normalize_address(b, c)
    rows.append({"S1 normalized": x.normalized, "target normalized": y.normalized, "hn": f"{x.house_number_core} | {y.house_number_core}",
                 "city": f"{x.city} | {y.city}", "state": f"{x.state} | {y.state}"})
display(pd.DataFrame(rows))

### Examples: Indic vs Latin true pairs after transliteration


,S1 raw,target raw,S1 core_latin,target core_latin,phonetic S1,phonetic target
0,Bright Food Private Limited,ब्राइट फूड प्राइवेट लिमिटेड,bright food,brait phud,prkt ft,prt ft
1,International Universal Power Private Limited,ইন্টারন্যাশনাল ইউনিভার্সাল পাওয়ার প্রাইভেট লিমিটেড,international universal power,intaranyashnal iunibharsal paoyar,antrntnl anprsl pr,antrnsnl anprsl pr
2,East Impex Private Limited,ઈસ્ટ ઇમ્પેક્સ પ્રાઇવેટ લિમિટેડ,east impex,ist impeks,ast ampks,ast ampks
3,Jai Marketing LLP,जय मार्केटिंग एलएलपी,jai marketing,jay marketing,j mrktnk,j mrktnk
4,Universal Prime Systems,యూనివర్సల్ ప్రైమ్ సిస్టమ్స్,universal prime systems,yunivarsal praim sistams,anprsl prm stms,anprsl prm stms
5,Tech Software Private Limited,টেক সফটওয়্যার প্রাইভেট লিমিটেড,tech software,tek saphatoyyar,tk sftpr,tk sftr
6,Urban Infotech Private Limited,അർബൻ ഇൻഫോടെക് പ്രൈവറ്റ് ലിമിറ്റഡ്,urban infotech,arban inphotek,arpn anftk,arpn anftk
7,Sree Media Private Limited,श्री मीडिया प्राइवेट लिमिटेड,sree media,shri midiya,sr mt,sr mt
8,Sunrise Enterprises Pvt Ltd,सनराइज एंटरप्राइजेज प्रा. लि.,sunrise enterprises,sanraij entarapraijej,snrs antrprs,snrj antrprj
9,Dynamic High Consultants Limited,डायनामिक हाई कंसल्टेंट्स लिमिटेड,dynamic high consultants,daynamik hai kansaltents,tnmk hk knsltnts,tnmk h knsltnts


### Examples: true pairs whose core_latin still disagrees (same script)


,S1,target,tset
0,properties agri center,properties center agri,100
1,mohamed plus,mplus,59
2,ay latin,service ay,44
3,richfield and sons,richfield,100
4,mckee advanced mfa,mckee aidveancde mfa,89
5,zeix holdings group,zeix holdings,100
6,global pacific montreal,pacificmontreal,47
7,4 corners,4 ccorners,95
8,hitech trading,hitech limited service,60
9,trusted all software,trusted all sotwatlre,93


### Examples: address structure on true pairs


,S1 normalized,target normalized,hn,city,state
0,"769 parker ave, aurora, il","769 parker ave, aurora, il",769 | 769,aurora | aurora,il | il
1,"40668 thomas ln, hempstead, tx","40668 thomas ln, tx, hempstead",40668 | 40668,hempstead | hempstead,tx | tx
2,"203, 2nd fl, brj chinoy mahmood complex, 1-7-347 to 349, parklane,...","tg, 203, 2nd fl, brj chinoy mahmood complex, 1-7-347 to 349, parka...",203 | 203,hyderabad | secunderabad,tg | tg
3,"45 radford st, yonkers, ny","45 radford st, yonkers, ny",45 | 45,yonkers | yonkers,ny | ny
4,"1477 1400 rd, prairie view, ks","1400 rd, prairie view city, ks",1477 | 1400,prairie view | prairie view,ks | ks
5,"sco-1, 2nd fl main market, sector-14, rohtak, hr","sco-1, 2nd fl main market, sector-14, rohtak, hr",1 | 1,rohtak | rohtak,hr | hr
6,"flat no 7, 1st fl, jor bagh market, new delhi, s delhi, dl","flat no 7, 1st fl, jor bagh market, new delhi, s delhi, dl",7 | 7,s delhi | s delhi,dl | dl
7,"229 montana st, valier, mt","229 montana st, valier, mt",229 | 229,valier | valier,mt | mt


In [14]:
print("### Where house numbers disagree on true pairs (both present) — sample")
hd = pairs[(dt_true.addr_house_number_core_agree == False).values].sample(8, random_state=6)
display(pd.DataFrame([{"S1": a, "target": b, "hn S1": normalize_address(a, c).house_number, "hn target": normalize_address(b, c).house_number}
                      for a, b, c in zip(hd.a_addr, hd.b_addr, hd.country)]))

### Where house numbers disagree on true pairs (both present) — sample


,S1,target,hn S1,hn target
0,"460 2nd Street, Incorporated, ID","60 2ND ST, INCORPORATED, ID",460,60
1,"11509 Wynfair Court, Walton, KY","51509 WYNFAIR CT, WALTON, KY",11509,51509
2,"Gujarat, Rajkot, Office No. 809, 150 Feet Ring Road, Near Brts Bus...","ગુજરાત, THE SPIRE, OFFICE NO. 8-09, 150 FEET RING ROAD, NEAR BRTS ...",809,8-9
3,"2450 Water Street, Unit W8, Yuma, AZ","# W8, Yuma, ##2450 Water St, Arizona",2450,w8
4,"01/4, B/339, Dhakly Akhade Niwas, Shriram Lane, Worli Koliwada, Wo...","0-1/4, B/339, DHAKLY AKHADE NIWAS, SHRIRAM LANE, WORLI KOLIWADA, W...",1/4,0-1/4
5,"2Nd Floor, Hari Niwas Building 30Th Cross Road, Off Sv Road, Bandr...","Hari Niwas Building 30Th Cross Road, Off Sv Road, Bandra Wes, T, 1...",2nd,1-2nd
6,"1667 Bachan Court, Fairfax County, VA","166 BACHAN CT, FAIRFAX COUNTY, VA",1667,166
7,"Columbus, 108 Landers Avenue, OH","10 Landers Avenue, Ohio, Columbus",108,10


### 6.1 Key-agreement union (diagnostic only; this is not a candidate generator)
For several simple *agreement conditions* built from the Phase 2 representations, this cell reports the share of sampled true pairs where at least one condition holds, and the same share for random pairs.

These are **upper bounds on the recall** that exact keys of this kind could reach, measured on the sample. They say nothing about candidate volume, which Phase 3 must measure at full scale with DuckDB aggregations.

In [15]:
def conds(o):
    hn_state = (o.addr_house_number_core_agree == True) & (o.addr_state_agree == True)
    return pd.DataFrame({
        "K1 name variant equal (core_latin / alias / website)": o.name_variant_eq.astype(bool),
        "K2 state + house number equal": hn_state,
        "K3 shared core_latin token": o.name_shared_core_token.astype(bool),
        "K4 phonetic token_set >= 80": o.name_phonetic_ratio >= 80,
        "K5 shared place + shared number": (o.addr_shared_place == True) & (o.addr_shared_number == True),
    })
ct, cc = conds(dt_true), conds(dt_ctrl)
rows = []
for combo in [["K1"], ["K2"], ["K1", "K2"], ["K1", "K2", "K5"], ["K1", "K2", "K4"], ["K1", "K2", "K4", "K5"], ["K3"], ["K3", "K2"]]:
    cols = [c for c in ct.columns if c.split()[0] in combo]
    rows.append({"condition (OR)": " | ".join(combo), "true pairs %": round(100 * ct[cols].any(axis=1).mean(), 2),
                 "random pairs %": round(100 * cc[cols].any(axis=1).mean(), 2)})
union = pd.DataFrame(rows)
print("K1..K5 definitions:", *[f"  {c}" for c in ct.columns], sep="\n")
display(union)
miss = pairs[~ct[[c for c in ct.columns if c.split()[0] in ("K1", "K2", "K4", "K5")]].any(axis=1).values]
print(f"true pairs missed by K1|K2|K4|K5: {len(miss):,} — sample:")
display(miss.sample(min(8, len(miss)), random_state=8)[["src", "country", "a_name", "b_name", "a_addr", "b_addr"]])
RESULTS["key_agreement_union"] = union.to_dict(orient="records")

K1..K5 definitions:
  K1 name variant equal (core_latin / alias / website)
  K2 state + house number equal
  K3 shared core_latin token
  K4 phonetic token_set >= 80
  K5 shared place + shared number


,condition (OR),true pairs %,random pairs %
0,K1,58.30,0.01
1,K2,69.80,0.05
2,K1 | K2,86.68,0.05
3,K1 | K2 | K5,92.40,0.16
4,K1 | K2 | K4,97.56,0.25
5,K1 | K2 | K4 | K5,98.64,0.36
6,K3,87.96,1.53
7,K3 | K2,96.11,1.57


true pairs missed by K1|K2|K4|K5: 693 — sample:


,src,country,a_name,b_name,a_addr,b_addr
5421,S2,US,Rocky Motors LLC,ROCKY LLC (PARTNERS),"420 Rock Springs Drive, Marble Falls, TX","TX, MARBLE FALLS, ROCK SPRINGS DR"
2531,S3,India,Worth India Limited,Worth Límited Center,"Madhya Pradesh, Village Khejra, Guna, Guna, C/O Shivkumar Chandradas",NaN
44500,S2,India,Properties Karishma Estates Private Limited,@PROPERTIESKARISHMA,"Vill Kamdevpur P O Jagatballavpur, Hooghly, Howrah, West Bengal","#587 VILL KAMDEVPUR P O JAGATBALLAVPUR, HOOGHLY, West Bengal"
46229,S3,India,Seven Agro Pvt Ltd,Smt Seven Pvt Ltd Center,"Lodariyal Bavla Sanand Roadtal Sanand, Ahmedabad, Gujarat","220. Lodariyal Bavla Sanand Roadtal Sanand, Ahmedabad, GJ"
8774,S2,India,Eastern Producer Private Limited,Mr Eastern Private Limited Services,"11, B. B. Ganguly Street, Kolkata, Kolkata, Howrah, West Bengal",NaN
6737,S2,US,Hyacinth Costello Prime Gladstone LLC,#hyacinthcoste1lo,"405 Modesto Avenue, Russellville, AR","MODESTO AVENUE, RUSSELLVILLE, AR"
3857,S3,India,Fortune Management Private Limited,Fortune Private Limited (Services),"Ahmadabad City, 3Rd Floor, Gujarat, 312, Shaurya Icon, Narol-Vatva...","ગુજરાત, NULL, Floor"
2020,S2,India,Suryamukhi Veda Private Limited,Suryamukhi Private-Limited Services,"Flat No 302, Sai Srinivas, Srinivasa Nagar Colony, Qutubullapur, H...",NaN


## 7. Save results

In [16]:
RESULTS["runtime_total_sec"] = round(time.time() - T0, 1)
RESULTS["sample_sizes"] = {"records": int(len(smp)), "true_pairs": int(len(pairs)), "control_pairs": int(len(dt_ctrl))}
def _clean(o):
    if isinstance(o, dict): return {str(k): _clean(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)): return [_clean(v) for v in o]
    if isinstance(o, (np.integer,)): return int(o)
    if isinstance(o, (np.floating, float)): return None if np.isnan(o) else round(float(o), 4)
    if isinstance(o, np.bool_): return bool(o)
    return o
(EXP / "phase2_normalization_results.json").write_text(json.dumps(_clean(RESULTS), indent=2, ensure_ascii=False))
print("wrote experiments/phase2_normalization_results.json | total", RESULTS["runtime_total_sec"], "s")
con.close()

wrote experiments/phase2_normalization_results.json | total 83.5 s
